# 🚗 Indian License Plate Detector - YOLOv8n Training Pipeline (Colab T4 GPU)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/your-org/UrbanEye01/blob/main/anpr-training/anpr_plate_detection_colab.ipynb)

This notebook trains a high-speed, lightweight **YOLOv8n bounding box detector** for Indian vehicle license plates.

> **Important Scope Note**:
> - **This model is a bounding box detector only** (it detects the rectangle coordinates of the license plate).
> - **OCR is NOT trained here** — reading the alphanumeric characters is performed on-device inside the Android app using Google ML Kit Text Recognition.
> - After training, the model is exported to **ONNX format (`plate_detector.onnx`)** ready for the Android app assets folder.

## 1. Hardware & GPU Check
Verify that your Google Colab instance is connected to a GPU (go to **Runtime -> Change runtime type -> Hardware accelerator -> T4 GPU**).

In [ ]:
!nvidia-smi

## 2. Install Dependencies
Install `ultralytics` (YOLOv8), `roboflow` SDK, `onnx`, `onnxruntime`, and `opencv`.

In [ ]:
%pip install -q "ultralytics>=8.1.0" "roboflow>=1.1.0" "onnx>=1.15.0" "onnxruntime>=1.17.0" opencv-python matplotlib

## 3. Download Indian License Plate Dataset from Roboflow Universe

### Recommended Search Keywords on [Roboflow Universe](https://universe.roboflow.com):
1. `Indian license plate detection`
2. `ANPR India`
3. `Indian vehicle number plate`

### Option A: Download via Roboflow API (Recommended)
Replace the placeholders below with your credentials. You can find your API key under Roboflow Settings -> API Keys.

In [ ]:
# ==============================================================================
# ROBOFLOW CREDENTIALS (REPLACE PLACEHOLDERS)
# ==============================================================================
ROBOFLOW_API_KEY = "YOUR_ROBOFLOW_API_KEY"   # e.g., "aBcDeFg123456"
ROBOFLOW_WORKSPACE = "universe"             # or your personal workspace name
ROBOFLOW_PROJECT = "indian-license-plates"  # or specific project slug
ROBOFLOW_VERSION = 1                        # dataset version number
# ==============================================================================

import os
from pathlib import Path

if ROBOFLOW_API_KEY != "YOUR_ROBOFLOW_API_KEY":
    from roboflow import Roboflow
    rf = Roboflow(api_key=ROBOFLOW_API_KEY)
    project = rf.workspace(ROBOFLOW_WORKSPACE).project(ROBOFLOW_PROJECT)
    version = project.version(ROBOFLOW_VERSION)
    dataset = version.download("yolov8", location="./dataset")
    print(f"Dataset downloaded to: {dataset.location}")
else:
    print("[!] Please insert your Roboflow API key above, OR use Option B below to download manually via zip URL.")

### Option B: Manual Dataset Setup
If downloading via browser zip export from Roboflow Universe (Export -> Format: YOLOv8 -> download zip), upload `dataset.zip` to Colab and unzip it here:

In [ ]:
# Uncomment if you uploaded dataset.zip manually to Colab:
# !mkdir -p dataset
# !unzip -q dataset.zip -d dataset/
# !ls -la dataset/

## 4. Configure `plates.yaml`
Ensure `plates.yaml` points to the training data with a single class: `license_plate`.

In [ ]:
import yaml
from pathlib import Path

# If data.yaml was provided by Roboflow, inspect and adapt it
dataset_dir = Path("./dataset").resolve()
data_yaml_candidates = list(dataset_dir.glob("data.yaml")) + list(dataset_dir.glob("*.yaml"))

if data_yaml_candidates:
    with open(data_yaml_candidates[0], "r") as f:
        cfg = yaml.safe_load(f)
    
    cfg["path"] = str(dataset_dir)
    cfg["nc"] = 1
    cfg["names"] = {0: "license_plate"}
    
    with open("plates.yaml", "w") as f:
        yaml.dump(cfg, f, default_flow_style=False)
    print("Updated plates.yaml successfully from Roboflow export.")
else:
    # Default template
    template = f"""
path: {dataset_dir}
train: train/images
val: valid/images
test: test/images

nc: 1
names:
  0: license_plate
"""
    with open("plates.yaml", "w") as f:
        f.write(template.strip())
    print("Created plates.yaml template.")

!cat plates.yaml

## 5. Train YOLOv8n Plate Detector
We use the exact YOLOv8n architecture as the existing UrbanEye pothole/road defect model for consistency.

```bash
yolo detect train data=plates.yaml model=yolov8n.pt epochs=100 imgsz=640 batch=16
```

*On a Colab T4 GPU, 100 epochs typically finish in 15-20 minutes.*

In [ ]:
!yolo detect train data=plates.yaml model=yolov8n.pt epochs=100 imgsz=640 batch=16 project=runs/detect name=plate_train exist_ok=True

## 6. Evaluate Validation Metrics
Check mAP@0.5 and mAP@0.5:0.95 scores on the validation set.

In [ ]:
from ultralytics import YOLO

# Load the best trained model
best_model = YOLO("runs/detect/plate_train/weights/best.pt")
metrics = best_model.val(data="plates.yaml", imgsz=640)

print("=" * 60)
print("VALIDATION PERFORMANCE BENCHMARKS:")
print(f"  mAP@0.5      : {metrics.box.map50:.4f}")
print(f"  mAP@0.5:0.95 : {metrics.box.map:.4f}")
print(f"  Precision    : {metrics.box.mp:.4f}")
print(f"  Recall       : {metrics.box.mr:.4f}")
print("=" * 60)

## 7. Export Model to ONNX (`plate_detector.onnx`)
Exports matching UrbanEye's standard pattern (`best_float32.onnx`) with static input shape `[1, 3, 640, 640]` optimized for the Android ONNX Runtime.

In [ ]:
import shutil
from pathlib import Path

# Export command matching existing model pattern
!yolo export model=runs/detect/plate_train/weights/best.pt format=onnx imgsz=640 dynamic=False opset=12

# Rename to plate_detector.onnx and best_float32.onnx
src_onnx = Path("runs/detect/plate_train/weights/best.onnx")
shutil.copyfile(src_onnx, "plate_detector.onnx")
shutil.copyfile(src_onnx, "best_float32.onnx")

print(f"Model successfully exported: plate_detector.onnx ({Path('plate_detector.onnx').stat().st_size / (1024*1024):.2f} MB)")
print(f"Mirror copy created: best_float32.onnx")

## 8. Sanity-Check Visual Inference with ONNX Runtime
Run visual validation on sample validation images to verify detections before copying into Android.

In [ ]:
import cv2
import numpy as np
import onnxruntime as ort
import matplotlib.pyplot as plt
from pathlib import Path

# Load exported ONNX model
session = ort.InferenceSession("plate_detector.onnx", providers=["CPUExecutionProvider"])
input_name = session.get_inputs()[0].name

# Find sample test or val images
sample_images = list(Path("./dataset").glob("**/*.jpg"))[:3]

for img_p in sample_images:
    img_bgr = cv2.imread(str(img_p))
    if img_bgr is None:
        continue
    h, w = img_bgr.shape[:2]
    
    # Preprocessing: resize to 640x640, BGR->RGB, float32 [0, 1], CHW
    resized = cv2.resize(img_bgr, (640, 640))
    rgb = cv2.cvtColor(resized, cv2.COLOR_BGR2RGB)
    tensor = rgb.astype(np.float32) / 255.0
    tensor = np.transpose(tensor, (2, 0, 1))[None, ...]
    
    # Inference
    output = session.run(None, {input_name: tensor})[0]
    preds = output[0] if output.shape[1] > output.shape[2] else output[0].T
    
    # Filter conf >= 0.25
    boxes, scores = [], []
    for p in preds:
        score = float(p[4])
        if score >= 0.25:
            cx, cy, bw, bh = p[0] * (w / 640), p[1] * (h / 640), p[2] * (w / 640), p[3] * (h / 640)
            boxes.append([int(cx - bw/2), int(cy - bh/2), int(bw), int(bh)])
            scores.append(score)
            
    indices = cv2.dnn.NMSBoxes(boxes, scores, 0.25, 0.45)
    
    annotated = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
    if len(indices) > 0:
        for i in indices.flatten():
            bx, by, bw, bh = boxes[i]
            cv2.rectangle(annotated, (bx, by), (bx + bw, by + bh), (0, 255, 100), 3)
            cv2.putText(annotated, f"Plate {scores[i]:.1%}", (bx, max(20, by - 8)), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 255, 100), 2)
            
    plt.figure(figsize=(8, 6))
    plt.imshow(annotated)
    plt.axis("off")
    plt.title(f"{img_p.name} - Detections: {len(indices)}")
    plt.show()

## 9. Download the ONNX Model to Your Machine
Run this cell to download `plate_detector.onnx` to your computer, then copy it into the Android project assets folder:

`urbaneye-mobile/app/src/main/assets/models/plate_detector.onnx`

In [ ]:
from google.colab import files
files.download("plate_detector.onnx")